# Notebook Overview — Build Dataset (Optional)

## Purpose

This notebook provides an **optional, reproducible pipeline** for building a raw image dataset from **one selected source at a time**.

It retrieves candidate images, applies validation and deduplication, assigns standardized filenames, and generates raw metadata records for accepted images.

A **source reset mechanism** is included to allow clean rebuilding of any dataset source without manual intervention.

---

## Inputs

Supported dataset sources:

* ImageNet_1K_256
* MS_COCO_2017
* DiffusionDB
* SDXL_Generated_10K
* MidJourney
* OpenImages *(not supported in this notebook due to size)*

Only **one dataset source is processed per run**.

---

## Assumptions

* This notebook is **optional**
* Dataset construction is performed **one source at a time**
* Target-specific logic is implemented in:

  * `src/datasets/*.py`
* Accepted images must satisfy:

  * **width ≥ 256**
  * **height ≥ 256**
* Duplicate detection uses **SHA-256 hashes**
* Deduplication occurs at two levels:

  * per-source
  * global (cross-source)
* No preprocessing is performed here
* No dataset splitting is performed here

---

## Dataset Categories

| Target Name | Dataset            | Class Label | Dataset Code |
| ----------- | ------------------ | ----------- | ------------ |
| imagenet    | ImageNet_1K_256    | rl          | imgn         |
| coco        | MS_COCO_2017       | rl          | coco         |
| openimages  | OpenImages         | rl          | open         |
| diffusiondb | DiffusionDB        | ai          | diff         |
| sdxl        | SDXL_Generated_10K | ai          | sdxl         |
| midjourney  | MidJourney         | ai          | midj         |

---

## Outputs

For the selected dataset source:

### Images

Stored in:

```
data/raw/<SOURCE_DATASET>/images/
```

---

### Raw Metadata CSV

```
data/metadata/original/<dataset_code>_raw_metadata.csv
```

---

### Per-Source Hash File

```
data/metadata/hashes/<dataset_code>_source_hashes.json
```

---

### Global Hash File

```
data/metadata/hashes/global_hashes.json
```

---

## Expected Sizes

* ~3000 images per dataset source
* ~15,000 images across five Notebook 01 sources
* ~18,000 images across all project datasets

---

## Notes

* This notebook builds **raw datasets only**
* It is designed for **repeatable, deterministic execution**
* The reset mechanism allows rebuilding any source cleanly
* Deduplication ensures no overlap:

  * within a source
  * across sources
* This notebook does **not** perform:

  * preprocessing
  * feature extraction
  * dataset splitting

---
---

### Step 1 — Environment Setup and Configuration

* Clone GitHub repository if needed
* Load `project_config.py`
* Define notebook-level constants
* Ensure required directories exist

In [ ]:
# ============================================
# Step 1: Environment Setup and Configuration
# ============================================

import os
import sys
import importlib.util

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # User toggle (True or False)

# -------------------------------------------------
# Define project locations
# -------------------------------------------------
BASE_DIR = "/content/dip-ai-image-detection"
PROJECT_SRC_DIR = f"{BASE_DIR}/src"
CONFIG_FILE = f"{PROJECT_SRC_DIR}/project_config.py"

print("Initializing environment...")

# -------------------------------------------------
# Clone repo if not already present
# -------------------------------------------------
if not os.path.isdir(BASE_DIR):
    print("Cloning repository...")
    !git clone -q https://github.com/pgailinas/dip-ai-image-detection.git
else:
    print("Repository already exists. Skipping clone.")

# -------------------------------------------------
# Verify structure (always critical)
# -------------------------------------------------
if not os.path.isdir(PROJECT_SRC_DIR):
    raise FileNotFoundError(f"Missing directory: {PROJECT_SRC_DIR}")

if not os.path.isfile(CONFIG_FILE):
    raise FileNotFoundError(f"Missing file: {CONFIG_FILE}")

# -------------------------------------------------
# Add src to path
# -------------------------------------------------
sys.path.insert(0, PROJECT_SRC_DIR)

# -------------------------------------------------
# Import config
# -------------------------------------------------
spec = importlib.util.spec_from_file_location("project_config", CONFIG_FILE)
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)

# -------------------------------------------------
# Pull only what this notebook actually uses
# -------------------------------------------------
TARGET_COUNT          = cfg.TARGET_COUNT
BATCH_SIZE            = cfg.BATCH_SIZE
RANDOM_SEED           = cfg.RANDOM_SEED
RAW_DATA_DIR          = cfg.RAW_DATA_DIR
ORIGINAL_METADATA_DIR = cfg.ORIGINAL_METADATA_DIR
HASH_METADATA_DIR     = cfg.HASH_METADATA_DIR

# -------------------------------------------------
# Ensure required directories exist
# -------------------------------------------------
os.makedirs(cfg.DATA_DIR, exist_ok=True)
os.makedirs(cfg.RAW_DATA_DIR, exist_ok=True)
os.makedirs(cfg.METADATA_DIR, exist_ok=True)
os.makedirs(cfg.ORIGINAL_METADATA_DIR, exist_ok=True)
os.makedirs(cfg.HASH_METADATA_DIR, exist_ok=True)

# -------------------------------------------------
# Optional display block
# -------------------------------------------------
print("\nEnvironment setup complete.\n")

if VERBOSE:
    print(f"BASE_DIR          : {cfg.BASE_DIR}")
    print(f"DATA_DIR          : {cfg.DATA_DIR}")
    print(f"RAW_DATA_DIR      : {cfg.RAW_DATA_DIR}")
    print(f"METADATA_DIR      : {cfg.METADATA_DIR}")
    print(f"ORIGINAL_METADATA : {cfg.ORIGINAL_METADATA_DIR}")
    print(f"HASH_METADATA     : {cfg.HASH_METADATA_DIR}")
    print(f"TARGET_COUNT      : {TARGET_COUNT}")
    print(f"BATCH_SIZE        : {BATCH_SIZE}")
    print("\nOutputs will be written to local project directories.")



### Step 2 — Download Mode and Target Selection

* Select dataset source or skip Notebook 01
* Prevent unsupported OpenImages selection


In [ ]:
# ============================================
# Step 2: Download Mode and Target Selection
# ============================================

import ipywidgets as widgets
from IPython.display import display, clear_output

RUN_DOWNLOAD = False
TARGET_NAME = None

mode_widget = widgets.Dropdown(
    options=[
        ("Download one dataset", "download"),
        ("Skip Notebook 01", "skip"),
    ],
    value="download",
    description="Mode:"
)

dataset_widget = widgets.Dropdown(
    options=[
        ("ImageNet (SMALL ~331 MB)", "imagenet"),
        ("MS COCO 2017 (MEDIUM ~1.5 GB)", "coco"),
        ("DiffusionDB (MEDIUM ~1.8 GB)", "diffusiondb"),
        ("SDXL (VERY LARGE ~3.7 GB)", "sdxl"),
        ("MidJourney (VERY LARGE ~3.8 GB)", "midjourney"),
        ("OpenImages (EXTREME >20 GB — NOT SUPPORTED)", "openimages"),
    ],
    value="imagenet",
    description="Dataset:"
)

apply_button = widgets.Button(
    description="Apply Selection",
    button_style="primary"
)

output = widgets.Output()

def on_apply_clicked(b):
    global RUN_DOWNLOAD, TARGET_NAME

    with output:
        clear_output()

        if mode_widget.value == "skip":
            RUN_DOWNLOAD = False
            TARGET_NAME = None
            print("Notebook 01 skipped.")
            print("Proceed to 02_Preprocess_Images.ipynb.")
            return

        RUN_DOWNLOAD = True
        TARGET_NAME = dataset_widget.value

        if TARGET_NAME == "openimages":
            RUN_DOWNLOAD = False
            TARGET_NAME = None
            print("OpenImages is too large (>20 GB).")
            print("This dataset is not supported in Notebook 01.")
            print("Please select a different dataset or skip Notebook 01.")
            return

        size_map = {
            "imagenet": "SMALL (~331 MB)",
            "coco": "MEDIUM (~1.5 GB)",
            "diffusiondb": "MEDIUM (~1.8 GB)",
            "sdxl": "VERY LARGE (~3.7 GB)",
            "midjourney": "VERY LARGE (~3.8 GB)",
        }

        print("Selection Summary:")
        print(f"RUN_DOWNLOAD : {RUN_DOWNLOAD}")
        print(f"TARGET_NAME  : {TARGET_NAME}")
        print(f"Dataset Size : {size_map[TARGET_NAME]}")

apply_button.on_click(on_apply_clicked)

display(mode_widget)
display(dataset_widget)
display(apply_button)
display(output)



### Step 3 — Load Target Module

* Dynamically import dataset-specific module
* Resolve dataset name, label, and code


In [ ]:
# ============================================
# Step 3: Load Target Module
# ============================================

import importlib.util
import os

TARGET_SPECS = {
    "diffusiondb": {
        "display_name": "DiffusionDB",
        "module_name": "diffusiondb_target",
        "source_dataset": "DiffusionDB",
        "class_label": "ai",
        "dataset_code": "diff",
    },
    "sdxl": {
        "display_name": "SDXL Generated (10K)",
        "module_name": "sdxl_target",
        "source_dataset": "SDXL_Generated_10K",
        "class_label": "ai",
        "dataset_code": "sdxl",
    },
    "imagenet": {
        "display_name": "ImageNet (256)",
        "module_name": "imagenet_target",
        "source_dataset": "ImageNet_1K_256",
        "class_label": "rl",
        "dataset_code": "imgn",
    },
    "coco": {
        "display_name": "MS COCO 2017",
        "module_name": "coco_target",
        "source_dataset": "MS_COCO_2017",
        "class_label": "rl",
        "dataset_code": "coco",
    },
    "midjourney": {
        "display_name": "Midjourney",
        "module_name": "midjourney_target",
        "source_dataset": "Midjourney",
        "class_label": "ai",
        "dataset_code": "midj",
    },
    "openimages": {
        "display_name": "OpenImages",
        "module_name": "openimages_target",
        "source_dataset": "OpenImages",
        "class_label": "rl",
        "dataset_code": "open",
    },
}

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping module load (no valid dataset download selected).")

    TARGET_SPEC = None
    DISPLAY_NAME = None
    MODULE_NAME = None
    SOURCE_DATASET = None
    CLASS_LABEL = None
    DATASET_CODE = None
    target_module = None

else:
    if TARGET_NAME not in TARGET_SPECS:
        raise ValueError(f"Unsupported TARGET_NAME: {TARGET_NAME}")

    TARGET_SPEC = TARGET_SPECS[TARGET_NAME]

    DISPLAY_NAME   = TARGET_SPEC["display_name"]
    MODULE_NAME    = TARGET_SPEC["module_name"]
    SOURCE_DATASET = TARGET_SPEC["source_dataset"]
    CLASS_LABEL    = TARGET_SPEC["class_label"]
    DATASET_CODE   = TARGET_SPEC["dataset_code"]

    module_file = os.path.join(PROJECT_SRC_DIR, "datasets", f"{MODULE_NAME}.py")
    if not os.path.isfile(module_file):
        raise FileNotFoundError(f"Target module file not found: {module_file}")

    spec = importlib.util.spec_from_file_location(MODULE_NAME, module_file)
    target_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(target_module)

    print("Target module loaded successfully.")

    if VERBOSE:
        print()
        print(f"DISPLAY_NAME   : {DISPLAY_NAME}")
        print(f"MODULE_NAME    : {MODULE_NAME}")
        print(f"SOURCE_DATASET : {SOURCE_DATASET}")
        print(f"CLASS_LABEL    : {CLASS_LABEL}")
        print(f"DATASET_CODE   : {DATASET_CODE}")



### Step 4 — Common Configuration

* Define paths for:

  * raw image directory
  * metadata CSV
  * source hash JSON
  * global hash JSON

In [ ]:
# ============================================
# Step 4: Common Configuration
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping common configuration (no valid dataset download selected).")

    DATASET_RAW_DIR = None
    IMAGE_OUTPUT_DIR = None
    RAW_METADATA_CSV = None
    SOURCE_HASHES_JSON = None
    GLOBAL_HASHES_JSON = None

else:
    DATASET_RAW_DIR = os.path.join(cfg.RAW_DATA_DIR, SOURCE_DATASET)
    IMAGE_OUTPUT_DIR = os.path.join(DATASET_RAW_DIR, "images")

    RAW_METADATA_CSV = os.path.join(
        cfg.ORIGINAL_METADATA_DIR,
        f"{DATASET_CODE}_raw_metadata.csv"
    )

    SOURCE_HASHES_JSON = os.path.join(
        cfg.HASH_METADATA_DIR,
        f"{DATASET_CODE}_source_hashes.json"
    )

    GLOBAL_HASHES_JSON = os.path.join(
        cfg.HASH_METADATA_DIR,
        "global_hashes.json"
    )

    os.makedirs(DATASET_RAW_DIR, exist_ok=True)
    os.makedirs(IMAGE_OUTPUT_DIR, exist_ok=True)
    os.makedirs(cfg.ORIGINAL_METADATA_DIR, exist_ok=True)
    os.makedirs(cfg.HASH_METADATA_DIR, exist_ok=True)

    print("Common configuration complete.\n")

    if VERBOSE:
        print(f"DATASET_RAW_DIR    : {DATASET_RAW_DIR}")
        print(f"IMAGE_OUTPUT_DIR   : {IMAGE_OUTPUT_DIR}")
        print(f"RAW_METADATA_CSV   : {RAW_METADATA_CSV}")
        print(f"SOURCE_HASHES_JSON : {SOURCE_HASHES_JSON}")
        print(f"GLOBAL_HASHES_JSON : {GLOBAL_HASHES_JSON}")



### Step 5 — Reset Source State (Optional but Recommended)

* Removes current source hashes from global hash set
* Deletes:

  * source hash JSON
  * raw metadata CSV
  * source image directory
* Recreates required directories

This enables a **clean rebuild of the selected dataset source**.

In [ ]:
# ============================================
# Cell 5: Reset current source state for rebuild
# ============================================

RESET_SOURCE_FOR_REBUILD = True  # Set True only when you want to rebuild this source

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping source reset (no valid dataset selected).")

elif not RESET_SOURCE_FOR_REBUILD:
    print("Source reset skipped.")

else:
    import json
    import shutil

    print(f"Resetting saved state for current source: {SOURCE_DATASET}")

    # ----------------------------------------
    # Load current source hashes directly from file
    # ----------------------------------------
    if os.path.exists(SOURCE_HASHES_JSON):
        with open(SOURCE_HASHES_JSON, "r") as f:
            source_data = json.load(f)

        if isinstance(source_data, dict):
            source_hashes_reset = set(source_data.values())
        elif isinstance(source_data, list):
            source_hashes_reset = set(source_data)
        else:
            raise ValueError(f"Unsupported JSON structure in {SOURCE_HASHES_JSON}")
    else:
        source_hashes_reset = set()

    # ----------------------------------------
    # Load global hashes directly from file
    # ----------------------------------------
    if os.path.exists(GLOBAL_HASHES_JSON):
        with open(GLOBAL_HASHES_JSON, "r") as f:
            global_data = json.load(f)

        if isinstance(global_data, dict):
            global_hashes_reset = set(global_data.values())
        elif isinstance(global_data, list):
            global_hashes_reset = set(global_data)
        else:
            raise ValueError(f"Unsupported JSON structure in {GLOBAL_HASHES_JSON}")
    else:
        global_hashes_reset = set()

    original_source_count = len(source_hashes_reset)
    original_global_count = len(global_hashes_reset)

    # ----------------------------------------
    # Remove current source hashes from global
    # ----------------------------------------
    overlap = source_hashes_reset & global_hashes_reset
    global_hashes_reset -= overlap

    with open(GLOBAL_HASHES_JSON, "w") as f:
        json.dump(sorted(global_hashes_reset), f, indent=2)

    # ----------------------------------------
    # Delete current source hash file
    # ----------------------------------------
    if os.path.exists(SOURCE_HASHES_JSON):
        os.remove(SOURCE_HASHES_JSON)

    # ----------------------------------------
    # Delete current source metadata CSV
    # ----------------------------------------
    if os.path.exists(RAW_METADATA_CSV):
        os.remove(RAW_METADATA_CSV)

    # ----------------------------------------
    # Delete current source image directory
    # ----------------------------------------
    if os.path.exists(DATASET_RAW_DIR):
        shutil.rmtree(DATASET_RAW_DIR)

    # ----------------------------------------
    # Recreate required directories
    # ----------------------------------------
    os.makedirs(DATASET_RAW_DIR, exist_ok=True)
    os.makedirs(IMAGE_OUTPUT_DIR, exist_ok=True)
    os.makedirs(ORIGINAL_METADATA_DIR, exist_ok=True)
    os.makedirs(HASH_METADATA_DIR, exist_ok=True)

    print(f"Original source hash count : {original_source_count}")
    print(f"Original global hash count : {original_global_count}")
    print(f"Removed from global        : {len(overlap)}")
    print(f"New global hash count      : {len(global_hashes_reset)}")
    print("Source hash JSON deleted.")
    print("Source raw metadata CSV deleted.")
    print("Source image directory deleted and recreated.")
    print("Current source reset complete.")



### Step 6 — Raw Metadata CSV Setup

* Initialize metadata CSV with header
* Prepare for row-by-row appends



In [ ]:
# ============================================
# Step 6: Raw Metadata CSV Setup
# ============================================

import csv

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping raw metadata CSV setup (no valid dataset download selected).")
    RAW_METADATA_COLUMNS = None
    demo_metadata_rows = []

else:
    RAW_METADATA_COLUMNS = [
        "filename",
        "label",
        "dataset_code",
        "source_name",
        "source_id",
        "source_ref",
        "original_width",
        "original_height",
        "saved_path",
        "sha256",
        "batch_id",
    ]

    demo_metadata_rows = []

    if VERBOSE:
        if os.path.exists(RAW_METADATA_CSV):
            print("WARNING: Raw metadata CSV already exists and will be overwritten:")
            print(f"  {RAW_METADATA_CSV}")
        else:
            print("Creating new raw metadata CSV:")
            print(f"  {RAW_METADATA_CSV}")

    with open(RAW_METADATA_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(RAW_METADATA_COLUMNS)

    print("\nRaw metadata CSV initialized.")
    if VERBOSE:
        print("\nMetadata columns:")
        for col in RAW_METADATA_COLUMNS:
            print(f"  - {col}")



### Step 7 — Hash Utilities

* Load:

  * per-source hash set
  * global hash set
* Supports JSON formats:

  * list of hashes
  * filename → hash mapping


In [ ]:
# ============================================
# Step 7: Hash Utilities
# ============================================

import json

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping hash utilities setup (no valid dataset download selected).")

    source_hashes = set()
    global_hashes = set()

    def load_hash_set(json_path):
        return set()

    def save_hash_set(hash_set, json_path):
        return

else:
    def load_hash_set(json_path):
        if os.path.exists(json_path):
            with open(json_path, "r") as f:
                data = json.load(f)

            if isinstance(data, dict):
                return set(data.values())   # source hash JSON: filename -> hash
            elif isinstance(data, list):
                return set(data)            # global hash JSON: [hash, hash, ...]
            else:
                raise ValueError(f"Unsupported JSON structure in {json_path}")

        return set()

    def save_hash_set(hash_set, json_path):
        with open(json_path, "w") as f:
            json.dump(sorted(hash_set), f, indent=2)

    # Always load existing hashes if present
    source_hashes = load_hash_set(SOURCE_HASHES_JSON)
    global_hashes = load_hash_set(GLOBAL_HASHES_JSON)

    if VERBOSE:
        if os.path.exists(SOURCE_HASHES_JSON):
            print("Source hash file already exists and will be reused:")
            print(f"  {SOURCE_HASHES_JSON}")
        else:
            print("Source hash file will be created:")
            print(f"  {SOURCE_HASHES_JSON}")

        if os.path.exists(GLOBAL_HASHES_JSON):
            print("Global hash file already exists and will be reused:")
            print(f"  {GLOBAL_HASHES_JSON}")
        else:
            print("Global hash file will be created:")
            print(f"  {GLOBAL_HASHES_JSON}")

    print("\nLoaded hash counts:")
    print(f"  source_hashes : {len(source_hashes)}")
    print(f"  global_hashes : {len(global_hashes)}")



### Step 8 — Filename and Counting Utilities

* Count existing images
* Determine next index
* Generate standardized filenames:

  * `[dataset_code]_[index].png`

In [ ]:
# ============================================
# Step 8: Filename and Counting Utilities
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping filename and counting utilities (no valid dataset download selected).")

    def count_saved_images():
        return 0

    def get_next_image_index():
        return 1

    def make_filename(index):
        return None

else:
    def count_saved_images():
        if not os.path.exists(IMAGE_OUTPUT_DIR):
            return 0
        return len([
            f for f in os.listdir(IMAGE_OUTPUT_DIR)
            if f.lower().endswith(".png")
        ])

    def get_next_image_index():
        return count_saved_images() + 1

    def make_filename(index):
        return f"{DATASET_CODE}_{index:06d}.png"

    current_count = count_saved_images()
    next_index = get_next_image_index()
    sample_filename = make_filename(next_index)

    if VERBOSE:
        print(f"\nCurrent saved image count : {current_count}")
        print(f"Next image index          : {next_index}")

    print(f"Next filename example     : {sample_filename}")



### Step 9 — Image Utilities

* Normalize images to RGB
* Enforce minimum size constraint
* Compute SHA-256 hash from PNG byte content

In [ ]:
# ============================================
# Step 9: Image Utilities
# ============================================

import io
import hashlib
from PIL import Image

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping image utilities (no valid dataset download selected).")

    def normalize_to_rgb(image_obj):
        return None

    def meets_min_size(image_obj, min_size=256):
        return False

    def compute_image_sha256(image_obj):
        return None

else:
    def normalize_to_rgb(image_obj):
        """
        Convert input image to RGB PIL Image.
        """
        if image_obj.mode != "RGB":
            image_obj = image_obj.convert("RGB")
        return image_obj

    def meets_min_size(image_obj, min_size=256):
        """
        Return True if both width and height meet minimum size.
        """
        width, height = image_obj.size
        return width >= min_size and height >= min_size

    def compute_image_sha256(image_obj):
        """
        Compute SHA-256 hash from normalized PNG byte content.
        """
        image_obj = normalize_to_rgb(image_obj)

        buffer = io.BytesIO()
        image_obj.save(buffer, format="PNG")
        image_bytes = buffer.getvalue()

        return hashlib.sha256(image_bytes).hexdigest()

    print("Image utilities ready.\n")

    if VERBOSE:
        # Quick self-check using a synthetic image
        test_img = Image.new("RGB", (256, 256), color=(0, 0, 0))
        test_hash = compute_image_sha256(test_img)

        print(f"Minimum size check passed : {meets_min_size(test_img)}")
        print(f"Example SHA-256 length    : {len(test_hash)}")
        print(f"Example SHA-256 prefix    : {test_hash[:16]}...")



### Step 10 — CSV Append Utility

* Append metadata rows to CSV
* Ensures write consistency

In [ ]:
# ============================================
# Step 10: CSV Append Utility
# ============================================

import csv

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping CSV append utility (no valid dataset download selected).")

    demo_metadata_rows = []

    def append_metadata_row(row_dict):
        return

else:
    demo_metadata_rows = []

    def append_metadata_row(row_dict):
        """
        Append one accepted image metadata row to CSV.
        """
        with open(RAW_METADATA_CSV, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=RAW_METADATA_COLUMNS)
            writer.writerow(row_dict)

    print("CSV append utility ready.")

    if VERBOSE:
        print(f"Target CSV : {RAW_METADATA_CSV}")



### Step 11 — Load Source Dataset

* Load dataset via target module
* Initialize iteration state

In [ ]:
# ============================================
# Step 11: Load Source Dataset
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping source dataset load (no valid dataset download selected).")
    dataset_obj = None

else:
    print("Loading source dataset...\n")

    try:
        dataset_obj = target_module.load_source_dataset()
    except Exception as e:
        print("ERROR: Failed to load source dataset.")
        raise e

    print("Source dataset loaded successfully.\n")

    if VERBOSE:
        print(f"Dataset type : {type(dataset_obj)}")

    # Always reflect actual saved state
    current_count = count_saved_images()

    if VERBOSE:
        print(f"\nCurrent accepted images : {current_count}")



### Step 12 — Preview Candidate Records

* Retrieve sample records
* Display image and metadata information

In [ ]:
# ============================================
# Step 12: Preview Candidate Records
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping candidate preview (no valid dataset download selected).")

else:
    print("Retrieving candidate records...\n")

    try:
        # Get a small preview batch
        preview_records = target_module.get_candidate_records(dataset_obj, batch_size=3)
    except Exception as e:
        print("ERROR: Failed to retrieve candidate records.")
        raise e

    if not preview_records:
        print("No candidate records returned.")
    else:
        print(f"Retrieved {len(preview_records)} candidate records.")

        if VERBOSE:
            print()

            for i, record in enumerate(preview_records, start=1):
                print(f"Record {i}:")
                print(f"  source_id  : {record.get('source_id')}")
                print(f"  source_ref : {record.get('source_ref')}")

                img = record.get("image_obj")
                if img is not None:
                    try:
                        print(f"  image size : {img.size}")
                        print(f"  image mode : {img.mode}")
                    except Exception:
                        print("  image info : <unavailable>")
                else:
                    print("  image_obj  : None")

                print()



### Step 13 — Process One Candidate Record

* Validate image
* Compute hash
* Apply deduplication:

  * source-level
  * global-level
* Save accepted image
* Append metadata row


In [ ]:
# ============================================
# Step 13: Process One Candidate Record
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping candidate record processing (no valid dataset download selected).")

    def process_candidate_record(record, index):
        return False

else:
    def process_candidate_record(record, index):
        """
        Process a single candidate record.
        Returns True if accepted, False otherwise.
        """

        try:
            source_id = record.get("source_id")
            source_ref = record.get("source_ref")
            image_obj = record.get("image_obj")

            if image_obj is None:
                return False

            # Normalize
            image_obj = normalize_to_rgb(image_obj)

            # Size filter
            if not meets_min_size(image_obj):
                return False

            # Compute hash
            sha256 = compute_image_sha256(image_obj)

            # Deduplication
            if sha256 in source_hashes:
                return False

            if sha256 in global_hashes:
                return False

            # Generate filename
            filename = make_filename(index)

            # Save accepted image
            saved_path = os.path.join(IMAGE_OUTPUT_DIR, filename)
            image_obj.save(saved_path, format="PNG")

            # Update hash sets
            source_hashes.add(sha256)
            global_hashes.add(sha256)

            # Build metadata row
            row = {
                "filename": filename,
                "label": CLASS_LABEL,
                "dataset_code": DATASET_CODE,
                "source_name": SOURCE_DATASET,
                "source_id": source_id,
                "source_ref": source_ref,
                "original_width": image_obj.size[0],
                "original_height": image_obj.size[1],
                "saved_path": saved_path,
                "sha256": sha256,
                "batch_id": None,
            }

            append_metadata_row(row)

            return True

        except Exception:
            return False

    print("Candidate record processing function ready.")



### Step 14 — Build Dataset for Selected Source

* Iterate through dataset in batches
* Supports:

  * iterator-based datasets
  * start-index-based datasets
* Continue until target count is reached
* Track processed / accepted / rejected counts

In [ ]:
# ============================================
# Step 14: Build Dataset for Selected Source
# ============================================

from tqdm.notebook import tqdm
import inspect
import itertools

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping dataset build (no valid dataset download selected).")

else:
    def build_dataset_for_selected_source(target_count=TARGET_COUNT, batch_size=BATCH_SIZE, max_batches=None):
        """
        Build dataset for the selected source until target_count accepted images are reached
        or candidate records are exhausted.
        Supports:
          - target modules with start_index support
          - iterable datasets via notebook-side iteration
        """

        accepted_count = 0
        rejected_count = 0
        processed_count = 0
        batch_id = 0
        next_index = get_next_image_index()

        print("Starting dataset build...\n")
        print(f"Target source     : {SOURCE_DATASET}")
        print(f"Target count      : {target_count}")
        print(f"Batch size        : {batch_size}")

        sig = inspect.signature(target_module.get_candidate_records)
        supports_start_index = "start_index" in sig.parameters

        if supports_start_index:
            batching_mode = "module start_index"
            source_iter = None
            candidate_offset = 0
        else:
            batching_mode = "notebook iterator"
            source_iter = iter(dataset_obj)
            candidate_offset = None

        if VERBOSE:
            print(f"Batching mode     : {batching_mode}\n")

        pbar = tqdm(total=target_count, desc="Accepted Images", unit="img", disable=not VERBOSE)

        while accepted_count < target_count:
            batch_id += 1

            if max_batches is not None and batch_id > max_batches:
                print("Reached max_batches limit.")
                break

            try:
                if supports_start_index:
                    candidate_records = target_module.get_candidate_records(
                        dataset_obj,
                        start_index=candidate_offset,
                        batch_size=batch_size
                    )
                    candidate_offset += batch_size
                else:
                    raw_batch = list(itertools.islice(source_iter, batch_size))
                    if not raw_batch:
                        candidate_records = []
                    else:
                        candidate_records = target_module.get_candidate_records(
                            raw_batch,
                            batch_size=batch_size
                        )

            except Exception as e:
                print(f"ERROR: Failed to retrieve candidate records for batch {batch_id}.")
                raise e

            if not candidate_records:
                print("No more candidate records returned. Stopping build.")
                break

            batch_accepted = 0
            batch_rejected = 0

            for record in candidate_records:
                processed_count += 1

                accepted = process_candidate_record(record, next_index)

                if accepted:
                    accepted_count += 1
                    batch_accepted += 1
                    next_index += 1
                    pbar.update(1)
                else:
                    rejected_count += 1
                    batch_rejected += 1

                if accepted_count >= target_count:
                    break

            save_hash_set(source_hashes, SOURCE_HASHES_JSON)
            save_hash_set(global_hashes, GLOBAL_HASHES_JSON)

            if supports_start_index:
                offset_text = f"{candidate_offset:5d}"
            else:
                offset_text = "iter "

            if VERBOSE:
                print(
                    f"Batch {batch_id:03d} | "
                    f"offset: {offset_text} | "
                    f"processed: {len(candidate_records):4d} | "
                    f"accepted: {batch_accepted:4d} | "
                    f"rejected: {batch_rejected:4d} | "
                    f"total accepted: {accepted_count:4d}"
                )

            if len(candidate_records) < batch_size:
                print("Final partial batch reached. Stopping build.")
                break

        pbar.close()

        print("\nDataset build complete.\n")
        print(f"Total processed : {processed_count}")
        print(f"Total accepted  : {accepted_count}")
        print(f"Total rejected  : {rejected_count}")
        print(f"Images saved to : {IMAGE_OUTPUT_DIR}")
        print(f"Metadata CSV    : {RAW_METADATA_CSV}")

        return {
            "processed_count": processed_count,
            "accepted_count": accepted_count,
            "rejected_count": rejected_count,
            "batches_run": batch_id,
        }

    print("Dataset build function ready.")



### Step 15 — Verify Saved Output

* Compare:

  * image count
  * metadata row count
* Detect inconsistencies

In [ ]:
# ============================================
# Step 15: Verify Saved Output
# ============================================

import csv

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping output verification (no valid dataset download selected).")

else:
    def verify_saved_output():
        print("Verifying output...\n")

        image_count = count_saved_images()

        if os.path.exists(RAW_METADATA_CSV):
            with open(RAW_METADATA_CSV, "r", newline="") as f:
                reader = csv.reader(f)
                row_count = sum(1 for _ in reader) - 1  # exclude header
        else:
            row_count = 0

        print(f"Saved images       : {image_count}")
        print(f"Metadata CSV rows  : {row_count}")

        if image_count == row_count:
            print("\nVerification passed: image count matches metadata row count.")
        else:
            print("\nWARNING: image count does not match metadata row count.")

        return {
            "image_count": image_count,
            "metadata_row_count": row_count,
            "match": (image_count == row_count),
        }

    print("Output verification function ready.")



### Step 16 — Run Dataset Build and Verification

* Execute full dataset build
* Print summary statistics
* Run verification step

In [ ]:
# ============================================
# Step 16: Run Dataset Build and Verification
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping dataset build and verification (no valid dataset selected).")

else:
    print("Running dataset build...\n")

    # -------------------------------------------------
    # Run builder
    # -------------------------------------------------
    results = build_dataset_for_selected_source(
        target_count=TARGET_COUNT,
        batch_size=BATCH_SIZE
    )

    print("\nBuild Summary:")
    print(f"Processed : {results['processed_count']}")
    print(f"Accepted  : {results['accepted_count']}")
    print(f"Rejected  : {results['rejected_count']}")
    print(f"Batches   : {results['batches_run']}")

    print("\nRunning verification...\n")

    verification = verify_saved_output()

    print("\nVerification Summary:")
    for key, value in verification.items():
        print(f"{key} : {value}")



### Step 17 — Final Summary Report

* Display:

  * dataset information
  * total counts
  * acceptance rate
* Confirm completion

In [ ]:
# ============================================
# Step 17: Final Summary Report
# ============================================

import time

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("No dataset was built in this session.")

else:
    print("========== FINAL SUMMARY ==========\n")

    print(f"Source Dataset : {SOURCE_DATASET}")
    print(f"Dataset Code   : {DATASET_CODE}")

    # ----------------------------------------
    # Counts
    # ----------------------------------------
    image_count = count_saved_images()

    print(f"\nSaved Images   : {image_count}")
    print(f"Metadata CSV   : {RAW_METADATA_CSV}")
    print(f"Image Dir      : {IMAGE_OUTPUT_DIR}")

    print(f"Source Hashes  : {len(source_hashes)}")
    print(f"Global Hashes  : {len(global_hashes)}")

    # ----------------------------------------
    # Efficiency metrics (if results available)
    # ----------------------------------------
    try:
        processed = results["processed_count"]
        accepted  = results["accepted_count"]

        accept_rate = accepted / processed if processed > 0 else 0

        print("\n--- Performance ---")
        print(f"Total Processed : {processed}")
        print(f"Total Accepted  : {accepted}")
        print(f"Acceptance Rate : {accept_rate:.4f}")

    except Exception:
        print("\n(No performance stats available)")

    # ----------------------------------------
    # Completion message
    # ----------------------------------------
    print("===================================")
    print("\nNotebook 01 execution complete.")

